# 01 — Generate Data

Builds the human review pool and generates AI reviews from all 4 generators.

Run this on Colab with a T4 GPU. Mount Drive to persist `data/` between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ECS111FinalProject'
import os; os.makedirs(PROJECT_DIR, exist_ok=True)
%cd $PROJECT_DIR

In [15]:
# First time only: clone repo into the project dir, or rsync your local copy
#!git clone https://github.com/evanxliu1/ECS111FinalProject .
!pip install -q -r requirements.txt

In [22]:
# Set API keys (or load .env)
from google.colab import userdata
userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
#os.environ['HF_TOKEN'] = '...'

## 1. Build the human review pool

In [23]:
!python -m src.data.load_human --n 10000

All_Beauty: 100% 1250/1250 [00:00<00:00, 14713.39rev/s]
Books: 100% 1250/1250 [00:00<00:00, 9441.80rev/s]
Electronics: 100% 1250/1250 [00:00<00:00, 15773.75rev/s]
Home_and_Kitchen: 100% 1250/1250 [00:00<00:00, 16230.52rev/s]
Sports_and_Outdoors: 100% 1250/1250 [00:00<00:00, 16031.90rev/s]
Toys_and_Games: 100% 1250/1250 [00:00<00:00, 17326.11rev/s]
Pet_Supplies: 100% 1250/1250 [00:00<00:00, 16437.89rev/s]
Office_Products: 100% 1250/1250 [00:00<00:00, 15902.79rev/s]

Saved 10000 human reviews to data/raw/human_reviews.parquet
                                           review_id  ... generator
0  Toys_and_Games_AGKASBHYZPGTEPO6LWZPVJWB2BVA_14...  ...     human
1  Home_and_Kitchen_AFZUK3MTBIBEDQOPAK3OATUOUKLA_...  ...     human
2   Books_AFFZVSTUS3U2ZD22A2NPZSKOCPGQ_1528335058725  ...     human
3  Home_and_Kitchen_AFZUK3MTBIBEDQOPAK3OATUOUKLA_...  ...     human
4  Home_and_Kitchen_AHZ6XMOLEWA67S3TX7IWEXXGWSOA_...  ...     human

[5 rows x 9 columns]

Category counts:
category
Toys_and_Game

## 2. Generate GPT-5 mini reviews (OpenAI API, paid ~$1-3)

In [ ]:
!python -m src.data.generate_openai --n 2500 --model gpt-5-mini --out data/generated/gpt5mini.parquet

## 3. Generate Granite 4.1 8B reviews (local, 4-bit on T4)

In [ ]:
!python -m src.data.generate_local_hf --generator granite --n 2500

## 4. Generate Gemma-4 E4B-it reviews

In [ ]:
!python -m src.data.generate_local_hf --generator gemma --n 2500

## 5. Generate Qwen3.6 (small) reviews
If `Qwen/Qwen3.6-7B-Instruct` is not the right id, override with `--hf-id`.

In [ ]:
!python -m src.data.generate_local_hf --generator qwen --n 2500